What You'll Learn:

✅ RAG (Retrieval-Augmented Generation) - AI + document search \
✅ Vector stores (ChromaDB) - storing and searching documents \
✅ Embeddings - converting text to numbers for similarity search \
✅ Document loaders - reading PDFs, text files, web pages \
✅ Tool integration - giving agents access to external data \
✅ Combining conversational memory + retrieval

In [7]:
import sys
from pathlib import Path

# Setup paths
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from utils.env_loader import load_environment
load_environment()

print("="*80)
print("TUTORIAL 2: QUESTION ANSWERING AGENT WITH RAG")
print("="*80)
print("\n📚 New Concepts:")
print("  • RAG (Retrieval-Augmented Generation)")
print("  • Vector stores (ChromaDB)")
print("  • Embeddings")
print("  • Document loaders")
print("  • Tools in LangGraph")
print("\n💡 Use tutorial_2_questions.ipynb on the right for experiments!")
print("="*80)

TUTORIAL 2: QUESTION ANSWERING AGENT WITH RAG

📚 New Concepts:
  • RAG (Retrieval-Augmented Generation)
  • Vector stores (ChromaDB)
  • Embeddings
  • Document loaders
  • Tools in LangGraph

💡 Use tutorial_2_questions.ipynb on the right for experiments!


In [8]:
"""
Imports for RAG System
"""

from typing import Annotated, TypedDict, Sequence
import os

# LangGraph (from Tutorial 1)
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.sqlite import SqliteSaver

# LangChain Core
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, SystemMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.documents import Document

# LLM
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# NEW: Vector Store (for storing documents)
from langchain_community.vectorstores import Chroma

# NEW: Document Loaders (for reading files)
from langchain_community.document_loaders import (
    TextLoader,
    PyPDFLoader,
    WebBaseLoader
)

# NEW: Text Splitters (for chunking documents)
from langchain_text_splitters import RecursiveCharacterTextSplitter

# NEW: Tools (for giving agent abilities)
from langchain_core.tools import create_retriever_tool
from langgraph.prebuilt import ToolNode

print("✅ All imports successful!")
print("\n📦 New libraries:")
print("  • Chroma - Vector database for document storage")
print("  • Document Loaders - Read PDFs, text files, web pages")
print("  • Text Splitters - Break documents into chunks")
print("  • Tools - Give agent access to external capabilities")

✅ All imports successful!

📦 New libraries:
  • Chroma - Vector database for document storage
  • Document Loaders - Read PDFs, text files, web pages
  • Text Splitters - Break documents into chunks
  • Tools - Give agent access to external capabilities


In [8]:
#!pip install langchain langchain-anthropic langgraph python-dotenv pydantic langchain_community requests

In [9]:
"""
LESSON 1: RAG Architecture
"""

print("📖 LESSON 1: How RAG Works")
print("="*80)

print("""
RAG PIPELINE:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

SETUP PHASE (Do Once):
1. Load Documents
   ├─ PDFs: "annual_report.pdf"
   ├─ Text: "meeting_notes.txt"
   └─ Web: "https://docs.company.com"
   
2. Split into Chunks
   ├─ Chunk 1: "Q3 revenue was $10M..."
   ├─ Chunk 2: "Key customers include..."
   └─ Chunk 3: "Next quarter goals..."
   
3. Create Embeddings (convert text to numbers)
   ├─ Chunk 1 → [0.23, 0.45, 0.12, ...]
   ├─ Chunk 2 → [0.67, 0.89, 0.34, ...]
   └─ Chunk 3 → [0.11, 0.56, 0.78, ...]
   
4. Store in Vector Database (ChromaDB)
   └─ Ready for search!

QUERY PHASE (Every Question):
1. User asks: "What was Q3 revenue?"
   
2. Convert question to embedding
   ├─ "What was Q3 revenue?" → [0.25, 0.48, 0.15, ...]
   
3. Search vector database for similar chunks
   ├─ Find: Chunk 1 (similarity: 95%)
   └─ "Q3 revenue was $10M..."
   
4. Give relevant chunks to LLM
   ├─ System: "Answer using this context..."
   ├─ Context: "Q3 revenue was $10M..."
   └─ Question: "What was Q3 revenue?"
   
5. LLM generates answer
   └─ "According to the documents, Q3 revenue was $10M"

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
""")

print("\n✓ RAG combines LLM intelligence with YOUR documents")
print("\n💡 Experiment in questions notebook:")
print("   - What are embeddings?")
print("   - How does similarity search work?")

📖 LESSON 1: How RAG Works

RAG PIPELINE:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

SETUP PHASE (Do Once):
1. Load Documents
   ├─ PDFs: "annual_report.pdf"
   ├─ Text: "meeting_notes.txt"
   └─ Web: "https://docs.company.com"
   
2. Split into Chunks
   ├─ Chunk 1: "Q3 revenue was $10M..."
   ├─ Chunk 2: "Key customers include..."
   └─ Chunk 3: "Next quarter goals..."
   
3. Create Embeddings (convert text to numbers)
   ├─ Chunk 1 → [0.23, 0.45, 0.12, ...]
   ├─ Chunk 2 → [0.67, 0.89, 0.34, ...]
   └─ Chunk 3 → [0.11, 0.56, 0.78, ...]
   
4. Store in Vector Database (ChromaDB)
   └─ Ready for search!

QUERY PHASE (Every Question):
1. User asks: "What was Q3 revenue?"
   
2. Convert question to embedding
   ├─ "What was Q3 revenue?" → [0.25, 0.48, 0.15, ...]
   
3. Search vector database for similar chunks
   ├─ Find: Chunk 1 (similarity: 95%)
   └─ "Q3 revenue was $10M..."
   
4. Give relevant chunks to LLM
   ├─ System: "Answer using this context..."
   ├─ Context: "Q

In [10]:
"""
LESSON 2: Creating Sample Documents
"""

print("📖 LESSON 2: Sample Documents for Testing")
print("="*80)

# Create sample documents directory
docs_dir = project_root / "data" / "documents"
docs_dir.mkdir(parents=True, exist_ok=True)

# Sample Document 1: Company Info
company_info = """
TechCorp Inc. - Company Overview
================================

Founded: 2020
Headquarters: San Francisco, CA
Employees: 500+

Products:
---------
1. CloudAI Platform - AI infrastructure for enterprises
2. DataSync Pro - Real-time data synchronization
3. SecureNet - Enterprise security solutions

Q3 2024 Results:
----------------
Revenue: $15.2 million (up 45% YoY)
Active Customers: 250 enterprise clients
Key Wins: 
- Signed Fortune 500 deal with MegaCorp
- Expanded into European market
- Launched AI Assistant product

Q4 Goals:
---------
- Reach $20M revenue
- Hire 50 new engineers
- Launch mobile app
"""

# Sample Document 2: Product Documentation
product_docs = """
CloudAI Platform - User Guide
==============================

Getting Started:
----------------
1. Sign up at cloudai.techcorp.com
2. Create API key in dashboard
3. Install SDK: pip install cloudai-sdk
4. Start building!

Features:
---------
- Natural Language Processing
- Computer Vision
- Speech Recognition
- Sentiment Analysis
- Text Generation

Pricing:
--------
Starter: $99/month - 10K API calls
Professional: $499/month - 100K API calls
Enterprise: Custom pricing

Support:
--------
Email: support@techcorp.com
Phone: 1-800-TECH-AI
Hours: 24/7
"""

# Sample Document 3: Meeting Notes
meeting_notes = """
Engineering Team Meeting - Dec 15, 2024
========================================

Attendees: Alice (CTO), Bob (Lead), Carol (PM)

Agenda:
-------
1. Sprint Review
2. Q4 Planning
3. Hiring Updates

Key Decisions:
--------------
✓ Approved migration to Kubernetes
✓ Decided to use PostgreSQL for analytics
✓ Greenlit AI Assistant v2.0 project

Action Items:
-------------
- Alice: Finalize architecture design by Dec 20
- Bob: Interview 3 senior engineers this week
- Carol: Draft product roadmap for Q1 2025

Next Steps:
-----------
- Deploy beta to 10 pilot customers
- Gather feedback by Jan 15
- Public launch Feb 1, 2025
"""

# Save documents
(docs_dir / "company_info.txt").write_text(company_info)
(docs_dir / "product_docs.txt").write_text(product_docs)
(docs_dir / "meeting_notes.txt").write_text(meeting_notes)

print(f"✓ Created sample documents in: {docs_dir}")
print("\nDocuments created:")
print("  1. company_info.txt - Company overview and Q3 results")
print("  2. product_docs.txt - Product documentation")
print("  3. meeting_notes.txt - Engineering meeting notes")
print("\n💡 These simulate real company documents for testing")

📖 LESSON 2: Sample Documents for Testing
✓ Created sample documents in: /Users/kanderaolaxminarasimharao/Downloads/MyAgents-Git/MyAgents/data/documents

Documents created:
  1. company_info.txt - Company overview and Q3 results
  2. product_docs.txt - Product documentation
  3. meeting_notes.txt - Engineering meeting notes

💡 These simulate real company documents for testing


In [11]:
"""
LESSON 3: Loading and Chunking Documents
"""

print("📖 LESSON 3: Document Loading and Chunking")
print("="*80)

# Load documents
print("\n1️⃣ Loading documents...")
documents = []

for file in docs_dir.glob("*.txt"):
    loader = TextLoader(str(file))
    docs = loader.load()
    documents.extend(docs)
    print(f"   ✓ Loaded: {file.name} ({len(docs[0].page_content)} chars)")

print(f"\n   Total documents loaded: {len(documents)}")

# Split documents into chunks
print("\n2️⃣ Splitting into chunks...")
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,        # Each chunk ~500 characters
    chunk_overlap=50,      # 50 char overlap between chunks
    length_function=len,
    separators=["\n\n", "\n", " ", ""]
)

chunks = text_splitter.split_documents(documents)
print(f"   ✓ Created {len(chunks)} chunks")

# Show example chunk
print("\n3️⃣ Example chunk:")
print("   " + "-"*76)
print(f"   {chunks[0].page_content[:200]}...")
print("   " + "-"*76)

print("\n💡 Why chunking?")
print("   • LLMs have token limits (can't process entire books)")
print("   • Smaller chunks = more precise retrieval")
print("   • Overlap prevents losing context at boundaries")

print("\n💡 Experiment in questions notebook:")
print("   - Try different chunk sizes")
print("   - See how overlap affects results")

📖 LESSON 3: Document Loading and Chunking

1️⃣ Loading documents...
   ✓ Loaded: product_docs.txt (554 chars)
   ✓ Loaded: meeting_notes.txt (625 chars)
   ✓ Loaded: company_info.txt (614 chars)

   Total documents loaded: 3

2️⃣ Splitting into chunks...
   ✓ Created 6 chunks

3️⃣ Example chunk:
   ----------------------------------------------------------------------------
   CloudAI Platform - User Guide

Getting Started:
----------------
1. Sign up at cloudai.techcorp.com
2. Create API key in dashboard
3. Install SDK: pip install cloudai-sd...
   ----------------------------------------------------------------------------

💡 Why chunking?
   • LLMs have token limits (can't process entire books)
   • Smaller chunks = more precise retrieval
   • Overlap prevents losing context at boundaries

💡 Experiment in questions notebook:
   - Try different chunk sizes
   - See how overlap affects results


In [12]:
"""
LESSON 4: Creating Vector Store with Embeddings
"""

print("📖 LESSON 4: Vector Store & Embeddings")
print("="*80)

print("\n1️⃣ Creating embeddings...")
print("   (Converting text chunks to numerical vectors)")

# Initialize embeddings model
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

print("   ✓ Using OpenAI text-embedding-3-small")
print("     • Fast and efficient")
print("     • 1536 dimensions per embedding")

# Create vector store
print("\n2️⃣ Creating ChromaDB vector store...")

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=str(project_root / "data" / "chroma_db")
)

print("   ✓ Vector store created!")
print(f"   • Stored {len(chunks)} document chunks")
print(f"   • Location: {project_root / 'data' / 'chroma_db'}")

# Test similarity search
print("\n3️⃣ Testing similarity search...")
test_query = "What was Q3 revenue?"
results = vectorstore.similarity_search(test_query, k=2)

print(f"   Query: '{test_query}'")
print(f"   Found {len(results)} relevant chunks:\n")

for i, doc in enumerate(results, 1):
    print(f"   Result {i}:")
    print(f"   {doc.page_content[:150]}...")
    print()

print("\n💡 What just happened?")
print("   1. Query converted to embedding: [0.23, 0.45, ...]")
print("   2. Compared to all chunk embeddings")
print("   3. Found most similar chunks (cosine similarity)")
print("   4. Returned top 2 matches")

print("\n💡 Experiment in questions notebook:")
print("   - Try different queries")
print("   - Change k (number of results)")
print("   - See what chunks are retrieved")

📖 LESSON 4: Vector Store & Embeddings

1️⃣ Creating embeddings...
   (Converting text chunks to numerical vectors)
   ✓ Using OpenAI text-embedding-3-small
     • Fast and efficient
     • 1536 dimensions per embedding

2️⃣ Creating ChromaDB vector store...
   ✓ Vector store created!
   • Stored 6 document chunks
   • Location: /Users/kanderaolaxminarasimharao/Downloads/MyAgents-Git/MyAgents/data/chroma_db

3️⃣ Testing similarity search...
   Query: 'What was Q3 revenue?'
   Found 2 relevant chunks:

   Result 1:
   Q3 2024 Results:
----------------
Revenue: $15.2 million (up 45% YoY)
Active Customers: 250 enterprise clients
Key Wins: 
- Signed Fortune 500 deal wi...

   Result 2:
   Q3 2024 Results:
----------------
Revenue: $15.2 million (up 45% YoY)
Active Customers: 250 enterprise clients
Key Wins: 
- Signed Fortune 500 deal wi...


💡 What just happened?
   1. Query converted to embedding: [0.23, 0.45, ...]
   2. Compared to all chunk embeddings
   3. Found most similar chunks (cos

In [13]:
"""
LESSON 5: Creating a Retrieval Tool
"""

print("📖 LESSON 5: Retrieval Tool for Agent")
print("="*80)

# Create retriever from vector store
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}  # Return top 3 most relevant chunks
)

# Create retriever tool
retriever_tool = create_retriever_tool(
    retriever,
    name="search_company_docs",
    description="Search company documents for information about TechCorp, products, meetings, and Q3 results. Use this when you need to answer questions about company information."
)

print("✓ Created retriever tool: 'search_company_docs'")
print("\nTool Details:")
print(f"  Name: {retriever_tool.name}")
print(f"  Description: {retriever_tool.description}")
print(f"  Returns: Top 3 relevant document chunks")

print("\n💡 What is a Tool?")
print("   A tool gives the agent a new ABILITY")
print("   ")
print("   Without tool: Agent can only chat")
print("   With tool:    Agent can search documents!")
print("   ")
print("   Agent decides WHEN to use the tool based on the question")

# Test the tool
print("\n🧪 Testing the tool...")
test_query = "What are TechCorp's products?"
tool_results = retriever_tool.invoke(test_query)

print(f"\nQuery: '{test_query}'")
print(f"Tool returned: {len(tool_results)} chunks\n")
print("Preview:")
print(tool_results[:200] + "...")

print("\n💡 Experiment in questions notebook:")
print("   - Call the tool with different queries")
print("   - See what chunks it retrieves")
print("   - Understand tool vs direct retriever")

📖 LESSON 5: Retrieval Tool for Agent
✓ Created retriever tool: 'search_company_docs'

Tool Details:
  Name: search_company_docs
  Description: Search company documents for information about TechCorp, products, meetings, and Q3 results. Use this when you need to answer questions about company information.
  Returns: Top 3 relevant document chunks

💡 What is a Tool?
   A tool gives the agent a new ABILITY
   
   Without tool: Agent can only chat
   With tool:    Agent can search documents!
   
   Agent decides WHEN to use the tool based on the question

🧪 Testing the tool...

Query: 'What are TechCorp's products?'
Tool returned: 901 chunks

Preview:
TechCorp Inc. - Company Overview

Founded: 2020
Headquarters: San Francisco, CA
Employees: 500+

Products:
---------
1. CloudAI Platform - AI infrastructure for enterp...

💡 Experiment in questions notebook:
   - Call the tool with different queries
   - See what chunks it retrieves
   - Understand tool vs direct retriever


In [14]:
"""
LESSON 6: Agent State (Enhanced from Tutorial 1)
"""

print("📖 LESSON 6: Agent State with Tools")
print("="*80)

class AgentState(TypedDict):
    """
    State for RAG agent.
    
    Enhanced from Tutorial 1 with tool support.
    """
    messages: Annotated[Sequence[BaseMessage], add_messages]

print("✓ Agent state defined")

print("\n📝 State Fields:")
print("   • messages: Conversation history (same as Tutorial 1)")
print("   ")
print("   Note: Tool results are automatically added to messages!")
print("   ")
print("   Example flow:")
print("   1. User: 'What was Q3 revenue?'")
print("   2. Agent: Calls search_company_docs tool")
print("   3. Tool: Returns relevant chunks")
print("   4. Tool result added to messages automatically")
print("   5. Agent: Reads tool result and answers")

print("\n💡 Compare to Tutorial 1:")
print("   Tutorial 1: Just messages (chat only)")
print("   Tutorial 2: Messages + tools (chat + document search)")

📖 LESSON 6: Agent State with Tools
✓ Agent state defined

📝 State Fields:
   • messages: Conversation history (same as Tutorial 1)
   
   Note: Tool results are automatically added to messages!
   
   Example flow:
   1. User: 'What was Q3 revenue?'
   2. Agent: Calls search_company_docs tool
   3. Tool: Returns relevant chunks
   4. Tool result added to messages automatically
   5. Agent: Reads tool result and answers

💡 Compare to Tutorial 1:
   Tutorial 1: Just messages (chat only)
   Tutorial 2: Messages + tools (chat + document search)


In [15]:
"""
LESSON 7: Building RAG Agent
"""

print("📖 LESSON 7: RAG Agent with Tool Calling")
print("="*80)

# Initialize LLM with tools
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Bind tools to LLM
llm_with_tools = llm.bind_tools([retriever_tool])

print("✓ LLM configured with tools")
print(f"   Model: {llm.model_name}")
print(f"   Tools: {[retriever_tool.name]}")

# Define agent function
def agent(state: AgentState):
    """
    Agent that can use tools to search documents.
    
    Flow:
    1. Receive messages
    2. LLM decides: answer directly OR use tool
    3. If tool needed: LLM generates tool call
    4. Return response (with or without tool call)
    """
    
    messages = state["messages"]
    response = llm_with_tools.invoke(messages)
    
    return {"messages": [response]}

print("\n✓ Agent function created")

print("\n💡 How tool calling works:")
print("   ")
print("   User: 'What was Q3 revenue?'")
print("   ↓")
print("   LLM thinks: 'I need to search documents'")
print("   ↓")
print("   LLM generates: TOOL CALL to search_company_docs")
print("   ↓")
print("   Tool executes: Searches vector store")
print("   ↓")
print("   Tool returns: Relevant document chunks")
print("   ↓")
print("   LLM reads results: Formulates answer")
print("   ↓")
print("   Answer: 'According to Q3 results, revenue was $15.2M'")

📖 LESSON 7: RAG Agent with Tool Calling
✓ LLM configured with tools
   Model: gpt-4o-mini
   Tools: ['search_company_docs']

✓ Agent function created

💡 How tool calling works:
   
   User: 'What was Q3 revenue?'
   ↓
   LLM thinks: 'I need to search documents'
   ↓
   LLM generates: TOOL CALL to search_company_docs
   ↓
   Tool executes: Searches vector store
   ↓
   Tool returns: Relevant document chunks
   ↓
   LLM reads results: Formulates answer
   ↓
   Answer: 'According to Q3 results, revenue was $15.2M'


In [16]:
"""
LESSON 8: LangGraph with Tool Node
"""

print("📖 LESSON 8: Building Graph with Tools")
print("="*80)

# Create graph
workflow = StateGraph(AgentState)

# Add nodes
workflow.add_node("agent", agent)
workflow.add_node("tools", ToolNode([retriever_tool]))

# Set entry point
workflow.set_entry_point("agent")

# Add conditional edge
def should_continue(state: AgentState):
    """
    Decide whether to use tools or end.
    
    If last message has tool_calls → route to tools
    Otherwise → end
    """
    messages = state["messages"]
    last_message = messages[-1]
    
    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        return "tools"
    return END

workflow.add_conditional_edges(
    "agent",
    should_continue,
    {
        "tools": "tools",
        END: END
    }
)

# After tools, go back to agent
workflow.add_edge("tools", "agent")

print("✓ Graph structure created")

# Compile with memory (CORRECTED!)
print("\n📊 Setting up persistent memory...")

import sqlite3

db_path = str(project_root / "data" / "qa_agent.db")

# Create connection explicitly
conn = sqlite3.connect(db_path, check_same_thread=False)

# Create checkpointer from connection
memory = SqliteSaver(conn)

# Compile graph
app = workflow.compile(checkpointer=memory)

print(f"✓ Graph compiled with persistent memory")
print(f"  Database: {db_path}")

print("\n📊 Graph Structure:")
print("   ")
print("   START → [agent] ─┐")
print("                    ├─→ (tool needed?) → [tools] ─┐")
print("                    │                              │")
print("                    └─→ (no tool) → END            │")
print("                                                   │")
print("                    ┌──────────────────────────────┘")
print("                    │")
print("                    └─→ [agent] → (answer) → END")
print("   ")

print("\n💡 Key difference from Tutorial 1:")
print("   Tutorial 1: START → agent → END (straight line)")
print("   Tutorial 2: Agent can loop back through tools!")

print("\n💡 Experiment in questions notebook:")
print("   - Trace through the graph flow")
print("   - See when tools are called vs not called")

📖 LESSON 8: Building Graph with Tools
✓ Graph structure created

📊 Setting up persistent memory...
✓ Graph compiled with persistent memory
  Database: /Users/kanderaolaxminarasimharao/Downloads/MyAgents-Git/MyAgents/data/qa_agent.db

📊 Graph Structure:
   
   START → [agent] ─┐
                    ├─→ (tool needed?) → [tools] ─┐
                    │                              │
                    └─→ (no tool) → END            │
                                                   │
                    ┌──────────────────────────────┘
                    │
                    └─→ [agent] → (answer) → END
   

💡 Key difference from Tutorial 1:
   Tutorial 1: START → agent → END (straight line)
   Tutorial 2: Agent can loop back through tools!

💡 Experiment in questions notebook:
   - Trace through the graph flow
   - See when tools are called vs not called


In [17]:
"""
LESSON 9: Chat Interface for RAG Agent
"""

print("📖 LESSON 9: Using the RAG Agent")
print("="*80)

def chat(user_input: str, thread_id: str = "default", verbose: bool = False):
    """
    Chat with RAG agent.
    
    Args:
        user_input: Your question
        thread_id: Conversation ID
        verbose: Show tool calls and retrieved docs
    
    Returns:
        AI response
    """
    config = {"configurable": {"thread_id": thread_id}}
    
    # Invoke agent
    result = app.invoke(
        {"messages": [HumanMessage(content=user_input)]},
        config=config
    )
    
    # Extract response
    messages = result["messages"]
    ai_response = messages[-1].content
    
    # If verbose, show tool usage
    if verbose:
        print("\n" + "="*80)
        print("EXECUTION TRACE")
        print("="*80)
        
        for msg in messages:
            if hasattr(msg, "tool_calls") and msg.tool_calls:
                print("\n🔧 Tool Call:")
                for tool_call in msg.tool_calls:
                    print(f"   Tool: {tool_call['name']}")
                    print(f"   Query: {tool_call['args']}")
            
            elif msg.type == "tool":
                print("\n📄 Tool Result:")
                print(f"   {msg.content[:200]}...")
        
        print("\n" + "="*80)
    
    return ai_response

print("✓ Chat function ready")

print("\n💡 Usage:")
print("   response = chat('What was Q3 revenue?', verbose=True)")
print("   ")
print("   With verbose=True:")
print("   • Shows which tools were called")
print("   • Shows what documents were retrieved")
print("   • Shows the reasoning process")

📖 LESSON 9: Using the RAG Agent
✓ Chat function ready

💡 Usage:
   response = chat('What was Q3 revenue?', verbose=True)
   
   With verbose=True:
   • Shows which tools were called
   • Shows what documents were retrieved
   • Shows the reasoning process


In [18]:
"""
LESSON 10: Testing RAG Agent
"""

print("📖 LESSON 10: Testing Questions")
print("="*80)

thread = "demo"

# Test 1: Question requiring document search
print("\n" + "="*80)
print("TEST 1: Document-based Question")
print("="*80)

question1 = "What was TechCorp's Q3 2024 revenue?"
print(f"\n👤 User: {question1}")
response1 = chat(question1, thread, verbose=True)
print(f"\n🤖 AI: {response1}")

# Test 2: Follow-up question (uses memory)
print("\n\n" + "="*80)
print("TEST 2: Follow-up Question (Memory Test)")
print("="*80)

question2 = "What were the key wins that quarter?"
print(f"\n👤 User: {question2}")
response2 = chat(question2, thread, verbose=True)
print(f"\n🤖 AI: {response2}")

# Test 3: Product question
print("\n\n" + "="*80)
print("TEST 3: Product Information")
print("="*80)

question3 = "What products does TechCorp offer?"
print(f"\n👤 User: {question3}")
response3 = chat(question3, thread, verbose=True)
print(f"\n🤖 AI: {response3}")

# Test 4: General knowledge (no tool needed)
print("\n\n" + "="*80)
print("TEST 4: General Knowledge (No Tool)")
print("="*80)

question4 = "What is Python?"
print(f"\n👤 User: {question4}")
response4 = chat(question4, thread, verbose=False)
print(f"\n🤖 AI: {response4}")
print("\n   Note: Agent should answer without calling tool!")

print("\n\n" + "="*80)
print("✅ All tests complete!")
print("="*80)

print("\n💡 Observations:")
print("   • Agent uses tool when asking about company docs")
print("   • Agent uses memory for follow-up questions")
print("   • Agent answers general questions without tools")
print("   • This is intelligent tool usage!")

📖 LESSON 10: Testing Questions

TEST 1: Document-based Question

👤 User: What was TechCorp's Q3 2024 revenue?

EXECUTION TRACE

🔧 Tool Call:
   Tool: search_company_docs
   Query: {'query': 'TechCorp Q3 2024 revenue'}

📄 Tool Result:
   Q3 2024 Results:
----------------
Revenue: $15.2 million (up 45% YoY)
Active Customers: 250 enterprise clients
Key Wins: 
- Signed Fortune 500 deal with MegaCorp
- Expanded into European market
- Laun...


🤖 AI: TechCorp's Q3 2024 revenue was $15.2 million, which represents a 45% increase year-over-year.


TEST 2: Follow-up Question (Memory Test)

👤 User: What were the key wins that quarter?

EXECUTION TRACE

🔧 Tool Call:
   Tool: search_company_docs
   Query: {'query': 'TechCorp Q3 2024 revenue'}

📄 Tool Result:
   Q3 2024 Results:
----------------
Revenue: $15.2 million (up 45% YoY)
Active Customers: 250 enterprise clients
Key Wins: 
- Signed Fortune 500 deal with MegaCorp
- Expanded into European market
- Laun...


🤖 AI: The key wins for TechCorp in 

In [5]:
print("="*80)
print("🎉 TUTORIAL 2 COMPLETE!")
print("="*80)

print("\n✅ What You Learned:")
print("   1. ✓ RAG (Retrieval-Augmented Generation)")
print("   2. ✓ Vector stores (ChromaDB)")
print("   3. ✓ Embeddings and similarity search")
print("   4. ✓ Document loading and chunking")
print("   5. ✓ Tools in LangGraph")
print("   6. ✓ Tool calling and conditional routing")
print("   7. ✓ Combining memory + retrieval")

print("\n🎯 Key Concepts:")
print("   • RAG = LLM + Your Documents")
print("   • Embeddings = Text as numbers")
print("   • Vector store = Database for similarity search")
print("   • Tools = Agent capabilities")
print("   • Conditional edges = Smart routing")

print("\n💡 Real-World Applications:")
print("   • Customer support chatbots (search company docs)")
print("   • Research assistants (search papers)")
print("   • Internal knowledge bases (search wikis)")
print("   • Legal assistants (search case law)")
print("   • Medical Q&A (search medical literature)")

print("\n📚 Next Steps:")
print("   • Add more document types (PDFs, web pages)")
print("   • Implement multiple tools")
print("   • Add citations (show which docs used)")
print("   • Build a Streamlit UI")
print("   • Move to Tutorial 3: Multi-Agent Systems")

print("\n" + "="*80)

🎉 TUTORIAL 2 COMPLETE!

✅ What You Learned:
   1. ✓ RAG (Retrieval-Augmented Generation)
   2. ✓ Vector stores (ChromaDB)
   3. ✓ Embeddings and similarity search
   4. ✓ Document loading and chunking
   5. ✓ Tools in LangGraph
   6. ✓ Tool calling and conditional routing
   7. ✓ Combining memory + retrieval

🎯 Key Concepts:
   • RAG = LLM + Your Documents
   • Embeddings = Text as numbers
   • Vector store = Database for similarity search
   • Tools = Agent capabilities
   • Conditional edges = Smart routing

💡 Real-World Applications:
   • Customer support chatbots (search company docs)
   • Research assistants (search papers)
   • Internal knowledge bases (search wikis)
   • Legal assistants (search case law)
   • Medical Q&A (search medical literature)

📚 Next Steps:
   • Add more document types (PDFs, web pages)
   • Implement multiple tools
   • Add citations (show which docs used)
   • Build a Streamlit UI
   • Move to Tutorial 3: Multi-Agent Systems

